# Loading, importing and configuration

In [ ]:
#install all required packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

#list of required packages
packages = [
    'groq',
    'sentence-transformers',  #for embeddings
    'numpy',
    'scikit-learn',  #for cosine similarity
    'torch',
    'transformers'
]

for package in packages:
    install_package(package)
    print(f"{package} installed")

In [ ]:
#import all libraries
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq
import json
import re
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#groq
GROQ_API_KEY = "enter_your_api_key_here"

#initialize groq client
if not GROQ_API_KEY:
    print("Please add your Groq API key in the GROQ_API_KEY variable")
    print("Get a free key at: https://groq.com")
else:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized successfully")

# CBT techniques

In [ ]:
#define cbt techniques
class CBTTechnique(Enum):
    COGNITIVE_RESTRUCTURING = "cognitive_restructuring"
    BEHAVIORAL_ACTIVATION = "behavioral_activation"
    GROUNDING = "grounding"
    PROBLEM_SOLVING = "problem_solving"
    SOCRATIC_QUESTIONING = "socratic_questioning"
    THOUGHT_CHALLENGING = "thought_challenging"
    MINDFULNESS = "mindfulness"
    EMOTION_REGULATION = "emotion_regulation"

#technique profiles for semantic matching
#these descriptions help the embedding model understand when to use each technique
TECHNIQUE_PROFILES = {
    CBTTechnique.COGNITIVE_RESTRUCTURING: {
        "description": "Challenge and reframe negative automatic thoughts, identify cognitive distortions",
        "example_phrases": [
            "I'm a complete failure",
            "Everyone hates me",
            "I'll never succeed",
            "I always mess up",
            "Nobody understands me",
            "I'm worthless",
            "Everything is terrible"
        ],
        "indicators": ["absolute thinking", "negative self-talk", "catastrophizing", "overgeneralization"]
    },

    CBTTechnique.BEHAVIORAL_ACTIVATION: {
        "description": "Increase engagement in meaningful activities to improve mood",
        "example_phrases": [
            "I can't get out of bed",
            "I have no motivation",
            "Nothing brings me joy anymore",
            "I'm too depressed to do anything",
            "I've stopped doing things I enjoy",
            "I feel stuck and inactive",
            "No energy for anything"
        ],
        "indicators": ["low mood", "lack of motivation", "withdrawal", "anhedonia"]
    },

    CBTTechnique.GROUNDING: {
        "description": "Anchor to present moment during anxiety or panic",
        "example_phrases": [
            "I can't breathe",
            "My heart is racing",
            "I'm having a panic attack",
            "I feel out of control",
            "Everything feels unreal",
            "I'm so anxious",
            "I might pass out"
        ],
        "indicators": ["panic", "dissociation", "acute anxiety", "physical symptoms"]
    },

    CBTTechnique.PROBLEM_SOLVING: {
        "description": "Structured approach to addressing practical problems",
        "example_phrases": [
            "I don't know what to do",
            "I need to figure this out",
            "I'm stuck with this problem",
            "How do I handle this situation",
            "I need help deciding",
            "What should I do about",
            "I'm facing a difficult choice"
        ],
        "indicators": ["decision-making", "practical issues", "seeking solutions"]
    },

    CBTTechnique.SOCRATIC_QUESTIONING: {
        "description": "Guide self-discovery through thoughtful questions",
        "example_phrases": [
            "I don't understand why I feel this way",
            "Why does this keep happening",
            "I want to understand myself better",
            "What's wrong with me",
            "I'm confused about my feelings"
        ],
        "indicators": ["self-exploration", "insight-seeking", "confusion"]
    },

    CBTTechnique.EMOTION_REGULATION: {
        "description": "Manage intense emotions effectively",
        "example_phrases": [
            "I can't control my anger",
            "My emotions are overwhelming",
            "I feel too much",
            "I'm emotionally out of control",
            "I go from 0 to 100 instantly"
        ],
        "indicators": ["emotional intensity", "mood swings", "emotional dysregulation"]
    }
}

print(f"Defined {len(TECHNIQUE_PROFILES)} CBT techniques with profiles")

In [ ]:
#initialize sentence transformer for embeddings
print("="*50)
print("Loading embedding model")
print("="*50)

#using a smaller, efficient model that works well for semantic similarity
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

#pre-compute embeddings for all technique example phrases
technique_embeddings = {}

for technique, profile in TECHNIQUE_PROFILES.items():
    #combine description and example phrases for embedding
    texts_to_embed = [profile["description"]] + profile["example_phrases"]

    #compute embeddings
    embeddings = embedding_model.encode(texts_to_embed)

    #store mean embedding for the technique
    technique_embeddings[technique] = np.mean(embeddings, axis=0)

print(f"Embedding model loaded and {len(technique_embeddings)} technique embeddings computed")

In [ ]:
#embedding-based technique selection
def select_technique_by_embedding(user_input: str, top_k: int = 3) -> List[Tuple[CBTTechnique, float]]:
    """
    Select CBT techniques using semantic similarity
    Returns top_k techniques with confidence scores
    """
    #encode user input
    user_embedding = embedding_model.encode([user_input])[0]

    #calculate similarities with all techniques
    similarities = {}
    for technique, technique_embedding in technique_embeddings.items():
        #cosine similarity between user input and technique
        similarity = cosine_similarity(
            user_embedding.reshape(1, -1),
            technique_embedding.reshape(1, -1)
        )[0][0]
        similarities[technique] = similarity

    #sort by similarity and get top k
    sorted_techniques = sorted(
        similarities.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    return sorted_techniques

#test embedding selection
test_input = "I feel like everything I do fails"
results = select_technique_by_embedding(test_input)
print(f"Test: '{test_input}'")
for technique, score in results:
    print(f"  {technique.value}: {score:.3f}")

In [ ]:
#rule-based validation system
class ValidationRule:
    """Base class for validation rules"""
    def __init__(self, name: str, priority: int = 5):
        self.name = name
        self.priority = priority  #1-10, higher = more important

    def validate(self, user_input: str, technique: CBTTechnique, context: Dict) -> Tuple[bool, str]:
        """
        Returns (is_valid, reason)
        """
        raise NotImplementedError

#safety rules
"""
#think is better to have a seperate logic to generate the response when user is at risk?
"""
class CrisisExclusionRule(ValidationRule):
    """Don't use certain techniques during crisis"""
    def __init__(self):
        super().__init__("crisis_exclusion", priority=10)
        self.crisis_keywords = ["suicide", "kill myself", "end my life", "want to die"]
        self.excluded_techniques = [
            CBTTechnique.BEHAVIORAL_ACTIVATION,
            CBTTechnique.PROBLEM_SOLVING
        ]

    def validate(self, user_input: str, technique: CBTTechnique, context: Dict) -> Tuple[bool, str]:
        #check for crisis keywords
        input_lower = user_input.lower()
        has_crisis = any(keyword in input_lower for keyword in self.crisis_keywords)

        if has_crisis and technique in self.excluded_techniques:
            return False, f"Cannot use {technique.value} during crisis"
        return True, "OK"

#appropriateness rules
class PanicGroundingRule(ValidationRule):
    """Must use grounding for panic symptoms"""
    def __init__(self):
        super().__init__("panic_grounding", priority=9)
        self.panic_indicators = ["can't breathe", "heart racing", "panic attack", "chest pain"]

    def validate(self, user_input: str, technique: CBTTechnique, context: Dict) -> Tuple[bool, str]:
        input_lower = user_input.lower()
        has_panic = any(indicator in input_lower for indicator in self.panic_indicators)

        if has_panic and technique != CBTTechnique.GROUNDING:
            return False, "Must use grounding for panic symptoms"
        return True, "OK"

#context rules
class EmotionalStateRule(ValidationRule):
    """Match technique to emotional state"""
    def __init__(self):
        super().__init__("emotional_state", priority=7)

    def validate(self, user_input: str, technique: CBTTechnique, context: Dict) -> Tuple[bool, str]:
        #simple emotion detection based on keywords
        input_lower = user_input.lower()

        #if expressing sadness/depression, prefer behavioral activation
        sadness_words = ["sad", "depressed", "hopeless", "empty"]
        if any(word in input_lower for word in sadness_words):
            if technique == CBTTechnique.BEHAVIORAL_ACTIVATION:
                return True, "Good match for depressive symptoms"

        #if expressing anxiety, prefer grounding or emotion regulation
        anxiety_words = ["anxious", "worried", "scared", "nervous"]
        if any(word in input_lower for word in anxiety_words):
            if technique in [CBTTechnique.GROUNDING, CBTTechnique.EMOTION_REGULATION]:
                return True, "Good match for anxiety"

        return True, "OK"

#initialize rule system
validation_rules = [
    CrisisExclusionRule(),
    PanicGroundingRule(),
    EmotionalStateRule()
]

print(f"Initialized {len(validation_rules)} validation rules")

In [ ]:
#hybrid technique selection combining embeddings and rules
class HybridTechniqueSelector:
    """Combines embedding similarity with rule-based validation"""

    def __init__(self):
        self.technique_history = []
        self.conversation_context = {}

    def select_technique(self, user_input: str) -> Tuple[CBTTechnique, Dict[str, Any]]:
        """
        Select best CBT technique using hybrid approach
        Returns (technique, selection_metadata)
        """
        #step 1: get top techniques from embeddings
        embedding_results = select_technique_by_embedding(user_input, top_k=3)

        #step 2: validate each technique with rules
        validation_results = []
        context = {
            "technique_history": self.technique_history,
            "conversation_context": self.conversation_context
        }

        for technique, similarity_score in embedding_results:
            #run all validation rules
            rule_violations = []
            total_priority = 0

            for rule in validation_rules:
                is_valid, reason = rule.validate(user_input, technique, context)
                if not is_valid:
                    rule_violations.append({
                        "rule": rule.name,
                        "reason": reason,
                        "priority": rule.priority
                    })
                    total_priority += rule.priority

            validation_results.append({
                "technique": technique,
                "similarity_score": similarity_score,
                "violations": rule_violations,
                "violation_score": total_priority,
                "is_valid": len(rule_violations) == 0
            })

        #step 3: select best valid technique
        #sort by: first validity, then similarity score
        valid_techniques = [r for r in validation_results if r["is_valid"]]

        if valid_techniques:
            #choose highest similarity among valid techniques
            best = max(valid_techniques, key=lambda x: x["similarity_score"])
            selected_technique = best["technique"]
        else:
            #if no valid techniques, choose one with least violations
            best = min(validation_results, key=lambda x: x["violation_score"])
            selected_technique = best["technique"]

        #update history
        self.technique_history.append(selected_technique)
        if len(self.technique_history) > 10:  #keep last 10
            self.technique_history.pop(0)

        #prepare metadata
        metadata = {
            "embedding_results": embedding_results,
            "validation_results": validation_results,
            "selected": selected_technique,
            "reasoning": self._generate_reasoning(validation_results, selected_technique)
        }

        return selected_technique, metadata

    def _generate_reasoning(self, validation_results: List, selected: CBTTechnique) -> str:
        """Generate explanation for technique selection"""
        for result in validation_results:
            if result["technique"] == selected:
                if result["is_valid"]:
                    return f"Selected {selected.value} (similarity: {result['similarity_score']:.3f})"
                else:
                    violations = ", ".join([v["reason"] for v in result["violations"]])
                    return f"Selected {selected.value} despite violations: {violations}"
        return f"Selected {selected.value}"

    def reset(self):
        """Reset conversation context"""
        self.technique_history = []
        self.conversation_context = {}

#initialize hybrid selector
hybrid_selector = HybridTechniqueSelector()

# Prompt Engineering

In [ ]:
#comprehensive cbt system prompt for groq
CBT_SYSTEM_PROMPT = """You are a supportive AI assistant trained in Cognitive Behavioral Therapy (CBT) techniques.

IMPORTANT: You will be told which specific CBT technique to apply. Follow that instruction.

CORE CBT PRINCIPLES:
1. Collaborative Empiricism: Work WITH the user to examine evidence objectively
2. Socratic Questioning: Guide discovery through thoughtful questions
3. Present-Focused: Concentrate on current problems and practical solutions
4. Cognitive Restructuring: Help identify and challenge unhelpful thought patterns
5. Behavioral Activation: Encourage engagement in meaningful activities

TECHNIQUE-SPECIFIC GUIDELINES:

For COGNITIVE_RESTRUCTURING:
- Identify the specific negative automatic thought
- Point out cognitive distortions (all-or-nothing, catastrophizing, etc.)
- Examine evidence for and against the thought
- Develop a balanced alternative thought

For BEHAVIORAL_ACTIVATION:
- Acknowledge low mood/lack of motivation
- Suggest small, achievable activities
- Focus on values-based actions
- Help schedule specific times

For GROUNDING:
- Guide through 5-4-3-2-1 sensory technique
- Encourage slow breathing
- Focus on immediate environment
- Provide calm reassurance

For PROBLEM_SOLVING:
- Help define the problem clearly
- Generate multiple solutions
- Evaluate pros and cons
- Create action steps

For SOCRATIC_QUESTIONING:
- Ask open-ended questions
- Explore beliefs and assumptions
- Guide self-discovery
- Avoid giving direct advice

For EMOTION_REGULATION:
- Validate the emotion
- Teach coping strategies
- Focus on emotion tolerance
- Suggest healthy outlets

RESPONSE STYLE:
- Be warm and empathetic
- Use "we" language for collaboration
- Keep responses focused and clear
- Never diagnose or replace professional therapy"""

print("CBT system prompt configured")

In [ ]:
#generate cbt response
def generate_cbt_response(
    user_input: str,
    technique: CBTTechnique,
    metadata: Dict[str, Any]
) -> Tuple[str, Dict]:
    """
    Generate CBT response using Groq with selected technique
    """
    #create technique-specific instruction
    technique_instruction = f"""
    Apply the {technique.value.replace('_', ' ').title()} technique.
    {TECHNIQUE_PROFILES[technique]['description']}

    User input: {user_input}
    """

    try:
        #call groq api
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",

            messages=[
                {"role": "system", "content": CBT_SYSTEM_PROMPT},
                {"role": "user", "content": technique_instruction}
            ],
            temperature=0.7,
            max_tokens=400
        )

        response_text = response.choices[0].message.content

        #add technique info to response
        enhanced_response = {
            "response": response_text,
            "technique": technique.value,
            "confidence": metadata["embedding_results"][0][1] if metadata["embedding_results"] else 0,
            "reasoning": metadata["reasoning"]
        }

        return response_text, enhanced_response

    except Exception as e:
        error_response = f"I apologize, I'm having trouble responding. Error: {str(e)}"
        return error_response, {"error": str(e)}

print("Response generation function configured")

In [ ]:
#complete cbt chatbot with hybrid technique selection
class CBTChatbot:
    """CBT Chatbot with hybrid embedding+rules technique selection"""

    def __init__(self):
        self.selector = HybridTechniqueSelector()
        self.conversation_history = []
        self.session_start = datetime.now()

    def respond(self, user_input: str) -> Dict[str, Any]:
        """
        Generate complete CBT response with technique selection
        """
        #select technique using hybrid approach
        technique, metadata = self.selector.select_technique(user_input)

        #generate response using groq
        response_text, response_data = generate_cbt_response(user_input, technique, metadata)

        #create complete response object
        result = {
            "user_input": user_input,
            "response": response_text,
            "technique": technique.value,
            "timestamp": datetime.now().isoformat(),
            "selection_metadata": metadata,
            "response_data": response_data
        }

        #update history
        self.conversation_history.append(result)

        return result

    def get_session_summary(self) -> Dict[str, Any]:
        """Get summary of conversation session"""
        if not self.conversation_history:
            return {"message": "No conversation yet"}

        #analyze techniques used
        techniques_used = [conv["technique"] for conv in self.conversation_history]
        technique_counts = {}
        for tech in techniques_used:
            technique_counts[tech] = technique_counts.get(tech, 0) + 1

        return {
            "session_duration": str(datetime.now() - self.session_start),
            "total_interactions": len(self.conversation_history),
            "techniques_used": technique_counts,
            "most_common_technique": max(technique_counts, key=technique_counts.get) if technique_counts else None
        }

    def reset(self):
        """Reset conversation"""
        self.selector.reset()
        self.conversation_history = []
        self.session_start = datetime.now()
        print("Conversation reset")

#initialize chatbot
chatbot = CBTChatbot()
print("CBT Chatbot with hybrid selection initialized")

In [ ]:
#test the hybrid system with various inputs
def test_hybrid_system():
    """Test the complete hybrid CBT system"""

    test_cases = [
        "I'm a complete failure at everything",
        "I can't breathe, my heart is racing",
        "Nothing brings me joy anymore",
        "I don't know how to solve this problem at work",
        "I want to understand why I feel this way"
    ]

    print("="*60)
    print("TESTING HYBRID CBT TECHNIQUE SELECTION")
    print("="*60)

    for i, test_input in enumerate(test_cases, 1):
        print(f"\n[Test {i}]")
        print(f"User: {test_input}")

        #get response
        result = chatbot.respond(test_input)

        #display results
        print(f"Selected Technique: {result['technique']}")
        print(f"Reasoning: {result['selection_metadata']['reasoning']}")
        print(f"Response: {result['response']}...")

        #show validation details
        print("\nValidation Details:")
        for val_result in result['selection_metadata']['validation_results'][:2]:
            tech = val_result['technique'].value
            score = val_result['similarity_score']
            valid = "VALID" if val_result['is_valid'] else "NOT VALID"
            print(f"  {tech}: similarity={score:.3f} {valid}")
            if val_result['violations']:
                for violation in val_result['violations']:
                    print(f"     {violation['reason']}")

        print("-"*60)

    #show session summary
    print("\n Session Summary:")
    summary = chatbot.get_session_summary()
    for key, value in summary.items():
        print(f"  {key}: {value}")

#run tests
test_hybrid_system()

In [ ]:
#interactive function for using the chatbot
def chat_with_cbt():
    """Interactive CBT chat session"""

    print("\n" + "="*60)
    print("CBT CHATBOT - Hybrid Technique Selection")
    print("="*60)
    print("Type 'quit' to exit | 'summary' for stats | 'reset' to start over")
    print("-"*60)

    while True:
        #get user input
        user_input = input("\nYou: ").strip()

        #handle commands
        if user_input.lower() == 'quit':
            print("\nTake care! Remember, professional support is always available.")
            break
        elif user_input.lower() == 'summary':
            summary = chatbot.get_session_summary()
            print("\nSession Summary:")
            for key, value in summary.items():
                print(f"  {key}: {value}")
            continue
        elif user_input.lower() == 'reset':
            chatbot.reset()
            continue

        #get response
        result = chatbot.respond(user_input)

        #display response
        print(f"\nCBT Bot [{result['technique']}]:")
        print(result['response'])

        #optionally show technique selection reasoning
        print(f"\nTechnique Selection: {result['selection_metadata']['reasoning']}")

#uncomment to start interactive chat
#chat_with_cbt()

print("\nChatbot ready! Call chat_with_cbt() to start interactive session")

In [ ]:
chat_with_cbt()